# Redes Cuánticas y QKD Avanzado

**Laboratorio 37 · Nivel muy avanzado**

Las redes cuánticas distribuyen entrelazamiento para comunicación segura,
computación distribuida y sensado de largo alcance.

Este laboratorio implementa y analiza:
- **BB84** con canal ruidoso: QBER y seguridad
- **E91** con prueba CHSH completa
- **MDI-QKD** (measurement-device-independent)
- **Repetidor cuántico** básico con purificación


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity
from qiskit.quantum_info import partial_trace

print('Entorno cargado OK')


## 1. BB84 con canal ruidoso: QBER y seguridad

BB84 es seguro si la **tasa de error cuántico de bit (QBER)** es baja:

$$\text{QBER} = \frac{\text{errores}}{\text{bits comparados}} < 11\%$$

Con Eve en modo intercept-and-resend, el QBER sube al 25%.


In [ ]:
class BB84Simulator:
    """Simulador de BB84 con canal binario simétrico (BSC) y Eve opcional."""

    def __init__(self, n_bits: int, p_channel: float, p_eve: float = 0.0,
                 seed: int = 42):
        """
        n_bits: número de bits enviados
        p_channel: probabilidad de error del canal (ruido)
        p_eve: fracción de bits interceptados por Eve
        """
        self.n = n_bits
        self.p_ch = p_channel
        self.p_eve = p_eve
        self.rng = np.random.default_rng(seed)

    def run(self) -> dict:
        rng = self.rng
        # Alice: bits aleatorios + bases aleatorias {Z, X}
        alice_bits  = rng.integers(0, 2, self.n)
        alice_bases = rng.integers(0, 2, self.n)  # 0=Z, 1=X
        bob_bases   = rng.integers(0, 2, self.n)

        # Eve: intercepta p_eve fracción de bits
        eve_intercepta = rng.random(self.n) < self.p_eve
        eve_bases       = rng.integers(0, 2, self.n)

        # Transmisión con canal ruidoso
        bob_recibidos = alice_bits.copy()
        # Errores de Eve
        for i in range(self.n):
            if eve_intercepta[i] and eve_bases[i] != alice_bases[i]:
                bob_recibidos[i] = rng.integers(0, 2)  # Eve reenvía con base aleatoria
        # Errores del canal
        canal_errors = rng.random(self.n) < self.p_ch
        bob_recibidos ^= canal_errors.astype(int)

        # Sifting: comparar bases
        bases_iguales = alice_bases == bob_bases
        alice_sifted = alice_bits[bases_iguales]
        bob_sifted   = bob_recibidos[bases_iguales]

        # QBER
        n_check = len(alice_sifted) // 4
        qber = np.mean(alice_sifted[:n_check] != bob_sifted[:n_check])

        # Clave final (eliminando bits de verificación)
        clave_alice = alice_sifted[n_check:]
        clave_bob   = bob_sifted[n_check:]

        return {
            'n_sifted':   len(alice_sifted),
            'n_clave':    len(clave_alice),
            'qber':       float(qber),
            'seguro':     qber < 0.11,
            'keyrate':    len(clave_alice) / self.n,
        }

# Sin Eve, distintos niveles de ruido
print('BB84 sin Eve:')
print(f'{"p_canal":>8} | {"QBER":>8} | {"Seguro":>8} | {"Key rate":>10}')
print('-' * 44)
for p in [0.0, 0.02, 0.05, 0.10, 0.15]:
    sim = BB84Simulator(10000, p_channel=p)
    res = sim.run()
    print(f'{p:>8.2f} | {res["qber"]:>8.4f} | {str(res["seguro"]):>8} | {res["keyrate"]:>10.4f}')

print('\nBB84 con Eve (intercept-and-resend 50% de bits):')
for p in [0.0, 0.02]:
    sim = BB84Simulator(10000, p_channel=p, p_eve=0.5)
    res = sim.run()
    print(f'  p_canal={p:.2f}: QBER={res["qber"]:.4f} (\'Detección de Eve: {"Sí" if res["qber"] > 0.11 else "No detectada"}\')')


In [ ]:
# Curva QBER vs Eve
p_eve_vals = np.linspace(0, 1, 50)
qbers = [BB84Simulator(20000, p_channel=0.01, p_eve=p, seed=i).run()['qber']
         for i, p in enumerate(p_eve_vals)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(p_eve_vals * 100, np.array(qbers) * 100, 'b-', lw=2)
ax.axhline(11, color='r', ls='--', lw=2, label='Umbral seguridad 11%')
ax.axvline(p_eve_vals[next(i for i,q in enumerate(qbers) if q > 0.11)] * 100,
           color='orange', ls=':', lw=2, label='Eve detectable')
ax.set_xlabel('Fracción de bits interceptados por Eve (%)')
ax.set_ylabel('QBER (%)')
ax.set_title('BB84: QBER vs actividad de Eve (p_canal=1%)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 2. E91: protocolo de Ekert con violación CHSH

E91 usa pares de Bell compartidos. La violación del desigualdad CHSH:
$$|E(a,b) - E(a,b') + E(a',b) + E(a',b')| \leq 2\quad \text{(clásico)}$$
$$|S_{\text{CHSH}}| \leq 2\sqrt{2} \approx 2.828\quad \text{(cuántico)}$$
garantiza la ausencia de variables ocultas locales (y la presencia de entrelazamiento).


In [ ]:
def correlacion_e91(theta_a: float, theta_b: float, F: float = 1.0) -> float:
    """
    Correlación E(θ_a, θ_b) para estado Bell Φ+ mezclado con ruido.
    F: fidelidad del estado (F=1 → Bell puro, F=0.5 → mezcla máxima).
    """
    # Estado mezclado: ρ = F|Φ+><Φ+| + (1-F)I/4
    # E(a,b) = F * (-cos(theta_a - theta_b)) para medidas en el plano XZ
    return -F * np.cos(theta_a - theta_b)

def chsh_parameter(F: float) -> float:
    """Parámetro CHSH óptimo para estado con fidelidad F."""
    # Ángulos óptimos: a=0, a'=pi/2, b=pi/4, b'=-pi/4
    a, a_p = 0, np.pi/2
    b, b_p = np.pi/4, -np.pi/4
    E = lambda ta, tb: correlacion_e91(ta, tb, F)
    return abs(E(a,b) - E(a,b_p) + E(a_p,b) + E(a_p,b_p))

F_vals = np.linspace(0.5, 1.0, 100)
S_vals = [chsh_parameter(F) for F in F_vals]

# Umbral de violación CHSH
F_threshold = next(F for F, S in zip(F_vals, S_vals) if S > 2.0)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(F_vals, S_vals, 'b-', lw=2, label='|S_CHSH|')
ax.axhline(2.0, color='r', ls='--', lw=2, label='Límite clásico = 2')
ax.axhline(2*np.sqrt(2), color='g', ls=':', lw=2, label=f'Máximo cuántico = 2√2 ≈ {2*np.sqrt(2):.3f}')
ax.axvline(F_threshold, color='orange', ls=':', lw=2, label=f'F_umbral = {F_threshold:.3f}')
ax.set_xlabel('Fidelidad del estado Bell'); ax.set_ylabel('|S_CHSH|')
ax.set_title('E91: violación CHSH vs pureza del estado')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Umbral de violación CHSH: F > {F_threshold:.3f}')
print(f'Para F<{F_threshold:.3f}: estado separable, no hay seguridad cuántica')


## 3. Repetidor cuántico: purificación

Para transmitir entrelazamiento sobre largas distancias, los repetidores cuánticos
dividen el canal en segmentos y purifican el entrelazamiento sucesivamente.

**Protocolo BBPSSW**: dado 2 pares de Bell ruidosos con fidelidad $F$,
se puede obtener 1 par con fidelidad $F' > F$ si $F > 1/2$.


In [ ]:
def fidelidad_tras_canal(F0: float, L: float, L_att: float = 22.0) -> float:
    """
    Fidelidad del estado Bell tras canal de fibra de longitud L km.
    L_att: longitud de atenuación (22 km para fibra estándar a 1550 nm).
    Modelo simplificado: F(L) = 0.5 + (F0-0.5)*exp(-L/L_att)
    """
    return 0.5 + (F0 - 0.5) * np.exp(-L / L_att)

def purificacion_bbpssw(F: float) -> float:
    """
    Purificación BBPSSW: 2 pares con fidelidad F → 1 par con fidelidad F'.
    Válido para F > 0.5.
    Probabilidad de éxito: P = F² + (1-F)²/3  (aproximado)
    """
    if F <= 0.5:
        return F  # no mejora
    num = F**2 + ((1-F)/3)**2
    den = F**2 + 2*F*(1-F)/3 + 5*((1-F)/3)**2
    return num / den

# Comparativa: con y sin repetidor para L=500 km
L_total = 500
F0 = 0.99  # fidelidad inicial del par generado

# Sin repetidor
F_directo = fidelidad_tras_canal(F0, L_total)

# Con 1 repetidor (2 segmentos de 250 km + 1 purificación)
F_seg1 = fidelidad_tras_canal(F0, L_total/2)
F_rep1 = purificacion_bbpssw(F_seg1)

# Con 3 repetidores (4 segmentos + 3 purificaciones)
F_seg_small = fidelidad_tras_canal(F0, L_total/4)
F_rep_round = F_seg_small
n_purif = 0
while F_rep_round < 0.95 and n_purif < 10:
    F_rep_round = purificacion_bbpssw(F_rep_round)
    n_purif += 1

print(f'Distancia: {L_total} km, F0 = {F0}')
print(f'Sin repetidor:  F = {F_directo:.6f}')
print(f'1 repetidor (250km/seg, 1 purif): F = {F_rep1:.6f}')
print(f'4 segmentos (125km/seg, {n_purif} purif): F = {F_rep_round:.6f}')

# Curva: fidelidad vs distancia
L_vals = np.linspace(0, 1000, 200)
F_sin_rep  = [fidelidad_tras_canal(F0, L) for L in L_vals]

def fidelidad_con_repetidores(L: float, n_seg: int) -> float:
    F = fidelidad_tras_canal(F0, L/n_seg)
    for _ in range(n_seg - 1):
        F_seg = fidelidad_tras_canal(F0, L/n_seg)
        F = purificacion_bbpssw(min(F, F_seg))
    return F

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(L_vals, F_sin_rep, 'r-', lw=2, label='Sin repetidor')
for n_seg, color in [(2,'b'), (4,'g'), (8,'purple')]:
    F_rep = [fidelidad_con_repetidores(L, n_seg) for L in L_vals]
    ax.plot(L_vals, F_rep, '-', lw=2, color=color, label=f'{n_seg-1} repetidores ({n_seg} segmentos)')
ax.axhline(0.9, color='gray', ls=':', lw=1, alpha=0.7)
ax.set_xlabel('Distancia (km)'); ax.set_ylabel('Fidelidad del estado Bell')
ax.set_title('Repetidores cuánticos: fidelidad vs distancia')
ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0.4, 1.0)
plt.tight_layout(); plt.show()


## 4. MDI-QKD: QKD sin confianza en los detectores

MDI-QKD (Lo, Curty, Qi 2012) elimina los ataques a los detectores del receptor.
Alice y Bob envían estados a un nodo central Charlie (no confiable) que realiza
una medida de Bell. La clave se genera solo con los casos exitosos.


In [ ]:
mdi_explicacion = """
MDI-QKD (Measurement-Device-Independent QKD)
════════════════════════════════════════════════════════

PROTOCOLO:
  1. Alice y Bob preparan estados cuánticos aleatoriamente (BB84 states).
  2. Ambos los envían a Charlie (nodo central, NO confiable).
  3. Charlie realiza una medida de Bell (BSM) y anuncia el resultado.
  4. Alice y Bob post-seleccionan en los casos donde la BSM fue exitosa.
  5. Sifting + estimación de errores + privacidad amplificación → clave.

VENTAJA SOBRE BB84/E91:
  • Inmune a todos los ataques a los detectores (side-channel attacks).
  • Charlie puede ser un adversario: no importa porque nunca ve la clave.
  • Demostrado experimentalmente (>400 km, 2020).

LIMITACIÓN:
  • Key rate más bajo: K ≈ Q²·H(F)  donde Q = eficiencia del canal
  • Requiere sincronización de fotones de Alice y Bob en Charlie
  • Doble canal de pérdida (Alice→Charlie + Bob→Charlie)

ESTADO DEL ARTE (2024):
  • TF-QKD (Twin-Field): supera el límite Pirandola-Laurenza-Ottaviani-Banchi
    con K ~ √η (√ de la transmisividad total), vs K ~ η² en MDI.
  • Record: 830 km por fibra óptica (2023, USTC China).

TASAS DE CLAVE TÍPICAS:
  Distancia | BB84    | MDI-QKD | TF-QKD
  50 km     | Mbps    | kbps    | kbps
  200 km    | kbps    | bps     | kbps
  500 km    | degradado| ×       | bps
  830 km    | ×        | ×       | <1 bps
"""
print(mdi_explicacion)

# Simulación de key rate vs distancia
L_vals = np.linspace(10, 600, 100)
eta_fiber = lambda L: 10**(-0.2 * L / 10)  # 0.2 dB/km a 1550 nm

def key_rate_bb84(L, eta_d=0.1, p_dark=1e-6):
    """Key rate normalizado para BB84 (bits/pulso)."""
    eta = eta_fiber(L) * eta_d
    Q = eta + p_dark
    if Q < 1e-12: return 0
    qber = p_dark / (2 * Q)
    if qber > 0.11: return 0
    return max(0, Q * (1 - 2 * max(0, -qber * np.log2(qber) - (1-qber) * np.log2(max(1e-30,1-qber)))))

def key_rate_mdi(L, eta_d=0.1):
    """Key rate MDI-QKD (proporcional a η² de cada brazo)."""
    eta = eta_fiber(L/2) * eta_d  # cada brazo L/2
    return max(0, eta**2 * 0.5)

def key_rate_tf(L, eta_d=0.1):
    """Key rate TF-QKD (proporcional a √η del canal total)."""
    eta_total = eta_fiber(L) * eta_d**2
    return max(0, np.sqrt(eta_total) * 0.01)

fig, ax = plt.subplots(figsize=(10, 5))
for func, nombre, color in [(key_rate_bb84,'BB84','b'),(key_rate_mdi,'MDI-QKD','r'),(key_rate_tf,'TF-QKD','g')]:
    rates = [func(L) for L in L_vals]
    rates_clean = [max(r, 1e-12) for r in rates]
    ax.semilogy(L_vals, rates_clean, '-', color=color, lw=2, label=nombre)
ax.set_xlabel('Distancia (km)'); ax.set_ylabel('Key rate (bits/pulso)')
ax.set_title('Comparativa QKD: tasas de clave vs distancia')
ax.legend(); ax.grid(alpha=0.3, which='both')
plt.tight_layout(); plt.show()


## 5. Entanglement Swapping: conectar nodos remotos

El *entanglement swapping* crea entrelazamiento entre nodos que nunca interactuaron
realizando una medida de Bell en los fotones intermedios.


In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def entanglement_swapping_circuit() -> QuantumCircuit:
    """
    Entanglement swapping:
    - Pares Bell: (0,1) y (2,3)
    - Medida de Bell en (1,2) → entrelaza (0,3)
    """
    qc = QuantumCircuit(4, 2)

    # Preparar 2 pares de Bell Φ+
    qc.h(0); qc.cx(0, 1)   # par 1: qubits 0,1
    qc.h(2); qc.cx(2, 3)   # par 2: qubits 2,3

    # Medida de Bell en qubits 1,2 (nodo intermedio Charlie)
    qc.cx(1, 2); qc.h(1)
    qc.measure(1, 0); qc.measure(2, 1)

    # Corrección clásica en qubit 3 basada en el resultado de Charlie
    with qc.if_else((qc.cregs[0][1], 1)):
        qc.x(3)
    with qc.if_else((qc.cregs[0][0], 1)):
        qc.z(3)

    return qc

# Versión sin corrección (para ver el estado entrelazado 0,3 condicionalmente)
qc_swap = QuantumCircuit(4)
qc_swap.h(0); qc_swap.cx(0, 1)
qc_swap.h(2); qc_swap.cx(2, 3)
qc_swap.cx(1, 2); qc_swap.h(1)

sv = Statevector(qc_swap)
state_arr = sv.data

# Los 4 componentes post-medida (proyectando en |00>, |01>, |10>, |11> en qubits 1,2)
print('Estado tras BSM en qubits 1,2 (sin corrección):')
for outcome in ['00', '01', '10', '11']:
    # Proyectar: qubit 1 = outcome[0], qubit 2 = outcome[1]
    idx_1 = int(outcome[0]); idx_2 = int(outcome[1])
    # Índice: qubit 3,2,1,0 → amplitudes donde bit1=idx_1 y bit2=idx_2
    amplitudes = np.array([state_arr[i] for i in range(16)
                           if (i >> 1) & 1 == idx_1 and (i >> 2) & 1 == idx_2])
    norma = np.linalg.norm(amplitudes)
    print(f'  |{outcome}>_12: P={norma**2:.4f}, estado_03 es Bell Φ+={np.allclose(norma, 0.5, atol=0.01)}')

print('\nResultado: qubits 0 y 3 quedan entrelazados con probabilidad 1,')
print('aunque nunca interactuaron directamente.')


## Conclusiones

- **BB84** es seguro para QBER < 11%; Eve con intercept-and-resend introduce QBER ≈ 25%.
- **E91** verifica seguridad mediante violación CHSH; requiere F > 0.5 del estado Bell.
- **Repetidores cuánticos** son esenciales para QKD > 300 km; MDI-QKD y TF-QKD extienden el alcance.
- **TF-QKD** rompe el límite de capacidad del canal con tasas $\propto \sqrt{\eta}$.
- **Entanglement swapping** es el componente fundamental de los repetidores cuánticos.
